# 01 · Exploratory Data Analysis: UCI Heart Disease

The figures are produced by `heart.eda` so the notebook, the CLI (`python -m heart.eda`) and the report all use identical plots.

Run `python -m heart.data` first if `data/processed/heart_clean.csv` does not exist.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from heart import config, eda
from heart.data import load_clean, missing_report

pd.set_option("display.precision", 2)
df = load_clean()
print(df.shape)
df.head()

## 1. Data quality
Missing values are kept as NaN in the cleaned file and imputed inside the model pipeline, so imputation statistics never leak from validation folds.

In [ ]:
display(missing_report(df))
display(df[config.INPUT_FEATURES].describe().T)
eda.plot_missing(df)
plt.show()

## 2. Class balance
_Write your observation: is the split balanced enough to use accuracy, or should ROC-AUC/recall lead?_

In [ ]:
eda.plot_class_balance(df)
plt.show()

## 3. Numeric features by diagnosis
The last panel shows the engineered `hr_reserve_ratio` = thalach / (220 − age). The dashed line marks 0.85, a common clinical cut-off for chronotropic incompetence.

_Observation:_

In [ ]:
eda.plot_numeric_distributions(df)
plt.show()

## 4. Disease rate within categorical levels
_Observation (e.g. asymptomatic chest pain, reversible thal defect, number of vessels):_

In [ ]:
eda.plot_categorical_rates(df)
plt.show()

## 5. Correlations
Spearman is used because several columns are ordinal codes.

_Observation: which features correlate most with the target? any strongly collinear pairs?_

In [ ]:
eda.plot_correlation(df)
plt.show()

In [ ]:
corr = df[config.INPUT_FEATURES + [config.TARGET]].corr(method="spearman")[config.TARGET]
corr.drop(config.TARGET).sort_values(key=abs, ascending=False)

## 6. Outliers
_Observation: are extreme cholesterol / BP values plausible patients or errors? (They are kept; the scaler and tree models are robust enough at this size.)_

In [ ]:
eda.plot_outliers(df)
plt.show()

## 7. Clinical risk bands
Disease rate across guideline bands for age (<45/45–54/55–64/65+), cholesterol (NCEP ATP III) and systolic BP (ACC/AHA 2017). These become the `clinical` feature set; the ablation in notebook 02 tests whether they add anything beyond the raw values.

_Observation:_

In [ ]:
eda.plot_clinical_bands(df)
plt.show()

## 8. Sex imbalance
The dataset is male-dominated, which is why the model is also evaluated separately for women and men.

_Observation:_

In [ ]:
eda.plot_sex_balance(df)
plt.show()

## 9. Save all figures for the report

In [ ]:
for p in eda.generate_all(df):
    print(p)

## Summary of findings
_Fill in 4–6 bullet points that justify the preprocessing and modelling choices._